In [ ]:
import os
import cv2
import supervision as sv
import ultralytics
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # For progress bar
from ultralytics import YOLO


# Change these paths and run all cells
model = YOLO("/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/runs_yolov12/runs/detect/train3/weights/best.pt")
dataset_path = "/home/omtalmo/Olaf_TTK4265/Poles/rgb"
output_dir = "/home/omtalmo/Olaf_TTK4265/TDT4265-Snow-pole-detection/results"

In [3]:
import time
import torch
import numpy as np
from tqdm.notebook import tqdm

def benchmark_model(model, test_images, num_runs=5, warm_up=2):
    """
    Benchmark YOLOv12 model inference speed.
    
    Args:
        model: YOLO model
        test_images: List of image paths to test
        num_runs: Number of runs to average performance over
        warm_up: Number of warm-up runs to exclude from timing
    
    Returns:
        DataFrame with performance metrics for each image
    """
    import pandas as pd
    
    # Results storage
    results = []
    
    # Ensure CUDA is properly warmed up if available
    device = next(model.parameters()).device
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    print(f"Running benchmark on {len(test_images)} images ({num_runs} runs each, {warm_up} warm-up runs)...")
    
    for image_idx, image_path in enumerate(tqdm(test_images)):
        image_name = Path(image_path).name
        
        # Load and prepare the image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Warning: Could not load {image_path}")
            continue
        
        # Warm-up runs
        for _ in range(warm_up):
            _ = model(image)
        
        # Actual timing runs
        times = []
        box_counts = []
        inference_times = []
        postprocessing_times = []
        total_boxes_processed = []
        
        for run in range(num_runs):
            # Start timing
            start_time = time.time()
            
            # Track internal metrics
            torch.cuda.synchronize() if device.type == 'cuda' else None
            inference_start = time.time()
            
            # Run detection
            with torch.no_grad():
                preds = model(image, verbose=False)
            
            # Synchronize before timing to ensure GPU operations complete
            torch.cuda.synchronize() if device.type == 'cuda' else None
            inference_end = time.time()
            
            # Get detections with NMS (this is the post-processing step)
            postprocessing_start = time.time()
            detections = sv.Detections.from_ultralytics(preds[0])
            postprocessing_end = time.time()
            
            # End timing
            end_time = time.time()
            
            # Record metrics
            total_time = end_time - start_time
            inference_time = inference_end - inference_start
            postprocessing_time = postprocessing_end - postprocessing_start
            
            # Count boxes
            boxes_before_nms = len(preds[0].boxes)
            boxes_after_nms = len(detections)
            
            times.append(total_time)
            inference_times.append(inference_time)
            postprocessing_times.append(postprocessing_time)
            box_counts.append(boxes_after_nms)
            total_boxes_processed.append(boxes_before_nms)
        
        # Calculate statistics
        avg_time = np.mean(times)
        std_time = np.std(times)
        avg_inference = np.mean(inference_times)
        avg_postprocessing = np.mean(postprocessing_times)
        avg_boxes = np.mean(box_counts)
        avg_boxes_processed = np.mean(total_boxes_processed)
        fps = 1.0 / avg_time if avg_time > 0 else 0
        
        # Store results
        results.append({
            'image': image_name,
            'avg_time_ms': avg_time * 1000,
            'std_time_ms': std_time * 1000,
            'inference_ms': avg_inference * 1000,
            'postprocessing_ms': avg_postprocessing * 1000,
            'fps': fps,
            'boxes_detected': avg_boxes,
            'boxes_processed': avg_boxes_processed
        })
        
    # Convert to DataFrame for easy analysis
    results_df = pd.DataFrame(results)
    
    # Print summary
    print("\nPerformance Summary:")
    print(f"Average FPS: {results_df['fps'].mean():.2f}")
    print(f"Average total time: {results_df['avg_time_ms'].mean():.2f} ms")
    print(f"Average inference time: {results_df['inference_ms'].mean():.2f} ms")
    print(f"Average postprocessing time: {results_df['postprocessing_ms'].mean():.2f} ms")
    print(f"Average number of boxes processed: {results_df['boxes_processed'].mean():.2f}")
    print(f"Average number of final detections: {results_df['boxes_detected'].mean():.2f}")
    
    return results_df

In [4]:
# Find test images with case-insensitive extension matching
test_image_dir = f"{dataset_path}/images/test"
test_images = []
for img in os.listdir(test_image_dir):
    if img.upper().endswith('.PNG') or img.lower().endswith(('.png', '.jpg', '.jpeg')):
        test_images.append(os.path.join(test_image_dir, img))


benchmark_model(model, test_images, num_runs=5, warm_up=2)

Running benchmark on 46 images (5 runs each, 2 warm-up runs)...


  0%|          | 0/46 [00:00<?, ?it/s]


0: 416x640 2 poles, 37.4ms
Speed: 1.7ms preprocess, 37.4ms inference, 74.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 poles, 5.2ms
Speed: 1.6ms preprocess, 5.2ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 640)

Performance Summary:
Average FPS: 177.73
Average total time: 5.63 ms
Average inference time: 5.56 ms
Average postprocessing time: 0.07 ms
Average number of boxes processed: 1.04
Average number of final detections: 1.04


,image,avg_time_ms,std_time_ms,inference_ms,postprocessing_ms,fps,boxes_detected,boxes_processed
0,frame_000005.PNG,6.055450,0.252304,5.954647,0.098848,165.140481,2.0,2.0
1,frame_000510.PNG,5.582380,0.373555,5.537796,0.043774,179.135055,0.0,0.0
2,frame_004760.PNG,5.939817,0.412318,5.873680,0.065041,168.355343,1.0,1.0
3,frame_000570.PNG,5.714560,0.162906,5.648899,0.064611,174.991614,1.0,1.0
4,frame_000315.PNG,5.838776,0.163043,5.758810,0.078201,171.268783,1.0,1.0
5,frame_004750.PNG,5.697060,0.111578,5.633211,0.062561,175.529144,1.0,1.0
6,frame_000260.PNG,5.690670,0.119055,5.599785,0.089359,175.726232,3.0,3.0
7,frame_001305.PNG,5.707407,0.161007,5.610037,0.095749,175.210915,2.0,2.0
8,frame_001580.PNG,5.391645,0.142006,5.350733,0.040054,185.472137,0.0,0.0
9,frame_003485.PNG,5.311728,0.051491,5.272579,0.038481,188.262669,0.0,0.0


In [6]:
# Function to create a performance comparison between original and optimized models
def compare_model_performance(original_model, optimized_model, test_images, num_images=5, runs=3):
    """Compare performance between the original model and the optimized model"""
    # Select a subset of test images for benchmarking
    benchmark_images = test_images[:num_images]
    
    print("Benchmarking original model...")
    original_results = benchmark_model(original_model, benchmark_images, num_runs=runs)
    
    print("\nBenchmarking optimized model...")
    optimized_results = benchmark_model(optimized_model, benchmark_images, num_runs=runs)
    
    # Create comparison visualization
    plt.figure(figsize=(12, 8))
    
    # FPS comparison
    plt.subplot(2, 2, 1)
    plt.bar(['Original', 'Optimized'], 
            [original_results['fps'].mean(), optimized_results['fps'].mean()], 
            color=['blue', 'green'])
    plt.title('Average FPS')
    plt.ylabel('Frames Per Second')
    
    # Timing breakdown
    plt.subplot(2, 2, 2)
    x = np.arange(2)
    width = 0.35
    plt.bar(x - width/2, 
            [original_results['inference_ms'].mean(), optimized_results['inference_ms'].mean()], 
            width, label='Inference')
    plt.bar(x + width/2, 
            [original_results['postprocessing_ms'].mean(), optimized_results['postprocessing_ms'].mean()], 
            width, label='Postprocessing')
    plt.xticks(x, ['Original', 'Optimized'])
    plt.title('Time Breakdown (ms)')
    plt.ylabel('Time (ms)')
    plt.legend()
    
    # Boxes processed
    plt.subplot(2, 2, 3)
    plt.bar(['Original', 'Optimized'], 
            [original_results['boxes_processed'].mean(), optimized_results['boxes_processed'].mean()], 
            color=['blue', 'green'])
    plt.title('Average Boxes Processed')
    plt.ylabel('Number of Boxes')
    
    # Final detections
    plt.subplot(2, 2, 4)
    plt.bar(['Original', 'Optimized'], 
            [original_results['boxes_detected'].mean(), optimized_results['boxes_detected'].mean()], 
            color=['blue', 'green'])
    plt.title('Average Final Detections')
    plt.ylabel('Number of Boxes')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'model_performance_comparison.png'))
    plt.show()
    
    # Calculate percentage improvements
    fps_improvement = (optimized_results['fps'].mean() / original_results['fps'].mean() - 1) * 100
    time_improvement = (1 - optimized_results['avg_time_ms'].mean() / original_results['avg_time_ms'].mean()) * 100
    box_reduction = (1 - optimized_results['boxes_processed'].mean() / original_results['boxes_processed'].mean()) * 100
    
    print(f"\nPerformance Improvements:")
    print(f"FPS increase: {fps_improvement:.1f}%")
    print(f"Total time reduction: {time_improvement:.1f}%")
    print(f"Box processing reduction: {box_reduction:.1f}%")
    
    return original_results, optimized_results